# Chapter 4: Pandas: loading, cleaning and transforming data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alilouati/programming-for-data-science/blob/main/04_pandas/lecture.ipynb)

**pandas** gives Python labelled tables (`DataFrame`) and columns (`Series`), with tools to read, clean, reshape, combine and summarise data. It was created by Wes McKinney in 2008, the author of *Python for Data Analysis*.

Why not just NumPy? Real data has

- **labels**: is column 2 `weight` or `weight2`?
- **mixed types**: names, dates, numbers in one table
- **missing and invalid values**: `NA`, `-1`, `-99999`
- **several tables** that must be aligned and joined

**Contents**

1. Series
2. DataFrame: creating, selecting, filtering
3. Reading and writing files
4. Inspecting a dataset
5. Cleaning: types, duplicates, missing and invalid values
6. Transforming: new columns, `apply`, `map`, `cut`, normalisation
7. Sorting and ranking
8. Grouping and aggregating
9. Combining tables: `merge` and `concat`
10. Reshaping: `melt`, `pivot_table`, MultiIndex
11. Working with dates and times

In [1]:
# Data files live in the course repository. This works both in Colab and locally.
from pathlib import Path

BASE_URL = "https://raw.githubusercontent.com/alilouati/programming-for-data-science/main/data/"

def data_file(name):
    """Return a local path if the repo is cloned, otherwise the GitHub URL."""
    local = Path("../data") / name
    return local if local.exists() else BASE_URL + name

In [2]:
import numpy as np
import pandas as pd
pd.__version__

'3.1.1'

## 1. Series

A `Series` is a one-dimensional array with an **index** (labels).

In [3]:
s = pd.Series([4, 7, -5, 3], index=["d", "b", "a", "c"])
print(s)
print(s["a"], s[["d", "c"]].tolist())
print(s[s > 0])
print(s * 2)

d    4
b    7
a   -5
c    3
dtype: int64
-5 [4, 3]
d    4
b    7
c    3
dtype: int64
d     8
b    14
a   -10
c     6
dtype: int64


A Series can be built from a dictionary. Operations **align on the index**; labels present in only one Series give `NaN` (missing).

In [4]:
population = pd.Series({"Riyadh": 7.0, "Jeddah": 3.8, "Dammam": 1.4, "Al-Kharj": np.nan}, name="population_m")
area = pd.Series({"Riyadh": 1973, "Jeddah": 1686, "Dammam": 800, "Taif": 321}, name="area_km2")
print(population.isna())
population * 1_000_000 / area

Riyadh      False
Jeddah      False
Dammam      False
Al-Kharj     True
Name: population_m, dtype: bool


Al-Kharj            NaN
Dammam      1750.000000
Jeddah      2253.855279
Riyadh      3547.896604
Taif                NaN
dtype: float64

## 2. DataFrame: creating, selecting, filtering

A `DataFrame` is a table: a dictionary of Series sharing one index.

In [5]:
df = pd.DataFrame({
    "name":   ["Nora", "Khalid", "Sara", "Omar", "Reem"],
    "gender": ["F", "M", "F", "M", "F"],
    "mid1":   [79, 95, 99, 76, 91],
    "mid2":   [95, 60, 67, 76, 84],
    "final":  [60, 61, 84, 97, 98],
})
df

,name,gender,mid1,mid2,final
0,Nora,F,79,95,60
1,Khalid,M,95,60,61
2,Sara,F,99,67,84
3,Omar,M,76,76,97
4,Reem,F,91,84,98


### Selecting columns and rows

| Syntax | Selects |
|---|---|
| `df["col"]` | one column, as a Series |
| `df[["a", "b"]]` | several columns, as a DataFrame |
| `df.loc[row_labels, col_labels]` | by **label** |
| `df.iloc[row_positions, col_positions]` | by **position** |
| `df[mask]` | rows where a boolean mask is True |

In [6]:
df["final"]

0    60
1    61
2    84
3    97
4    98
Name: final, dtype: int64

In [7]:
df = df.set_index("name")
df.loc["Sara"]

gender     F
mid1      99
mid2      67
final     84
Name: Sara, dtype: object

In [8]:
df.loc[["Nora", "Omar"], ["mid1", "final"]]

,mid1,final
name,,
Nora,79,60
Omar,76,97


In [9]:
df.iloc[:2, -1]

name
Nora      60
Khalid    61
Name: final, dtype: int64

In [10]:
df[(df["final"] > 80) & (df["gender"] == "F")]

,gender,mid1,mid2,final
name,,,,
Sara,F,99,67,84
Reem,F,91,84,98


In [11]:
df.query("final > 80 and gender == 'F'")    # same thing, often easier to read

,gender,mid1,mid2,final
name,,,,
Sara,F,99,67,84
Reem,F,91,84,98


### Adding, changing and dropping

In [12]:
df["average"] = df[["mid1", "mid2", "final"]].mean(axis=1).round(1)
df["passed"] = df["average"] >= 75
df.loc["Omar", "mid2"] = 80                     # change one value
df.drop(columns="passed")                        # returns a new DataFrame; df is unchanged

,gender,mid1,mid2,final,average
name,,,,,
Nora,F,79,95,60,78.0
Khalid,M,95,60,61,72.0
Sara,F,99,67,84,83.3
Omar,M,76,80,97,83.0
Reem,F,91,84,98,91.0


## 3. Reading and writing files

| Format | Read | Write |
|---|---|---|
| CSV / text | `pd.read_csv` | `df.to_csv` |
| Excel | `pd.read_excel` | `df.to_excel` |
| JSON | `pd.read_json` | `df.to_json` |
| Parquet (fast, typed, compressed) | `pd.read_parquet` | `df.to_parquet` |
| SQL database | `pd.read_sql` | `df.to_sql` |
| HTML tables on a web page | `pd.read_html` | `df.to_html` |

`read_csv` accepts a local path **or a URL**. Useful options: `sep`, `header`, `index_col`, `usecols`, `na_values`, `parse_dates`, `dtype`, `encoding`.

In [13]:
students = pd.read_csv(data_file("students.csv"))
courses = pd.read_csv(data_file("courses.csv"))
students.head()

,student_id,name,gender,program,study_hours_per_week,midterm1,midterm2,final
0,S2025000,Sara,F,MDS,6.5,71.0,54.0,64.0
1,S2025001,Maha,F,MIS,4.5,73.0,85.0,71.0
2,S2025002,Sara,F,MIS,1.7,57.0,51.0,42.0
3,S2025003,Maha,F,MIS,3.6,71.0,67.0,48.0
4,S2025004,Lama,F,MIS,7.2,93.0,78.0,-1.0


Writing is symmetric. `index=False` avoids saving the row numbers as an extra column.

In [14]:
from pathlib import Path
out = Path("output")
out.mkdir(exist_ok=True)
df.to_csv(out / "small_grades.csv")
pd.read_csv(out / "small_grades.csv", index_col="name").head(2)

,gender,mid1,mid2,final,average,passed
name,,,,,,
Nora,F,79,95,60,78.0,True
Khalid,M,95,60,61,72.0,False


JSON is the common format of web APIs:

In [15]:
import json
record = '{"name": "Nora", "programs": ["MDS"], "gpa": 4.6, "pet": null}'
print(json.loads(record))
print(df.head(2).to_json(orient="records"))

{'name': 'Nora', 'programs': ['MDS'], 'gpa': 4.6, 'pet': None}
[{"gender":"F","mid1":79,"mid2":95,"final":60,"average":78.0,"passed":true},{"gender":"M","mid1":95,"mid2":60,"final":61,"average":72.0,"passed":false}]


A small **SQLite** database, built into Python, shows the SQL round trip:

In [16]:
import sqlite3

with sqlite3.connect(":memory:") as con:
    students.to_sql("students", con, index=False)
    top = pd.read_sql(
        "SELECT program, COUNT(*) AS n, AVG(final) AS avg_final "
        "FROM students GROUP BY program ORDER BY n DESC", con)
top

,program,n,avg_final
0,MDS,50,69.860000
1,MIS,40,65.900000
2,MCS,28,60.892857
3,mds,4,69.250000


## 4. Inspecting a dataset

Always look before you analyse.

In [17]:
students.shape

(122, 8)

In [18]:
students.info()

<class 'pandas.DataFrame'>
RangeIndex: 122 entries, 0 to 121
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   student_id            122 non-null    str    
 1   name                  122 non-null    str    
 2   gender                122 non-null    str    
 3   program               122 non-null    str    
 4   study_hours_per_week  122 non-null    float64
 5   midterm1              122 non-null    float64
 6   midterm2              116 non-null    float64
 7   final                 122 non-null    float64
dtypes: float64(4), str(4)
memory usage: 7.8 KB


In [19]:
students.describe()

,study_hours_per_week,midterm1,midterm2,final
count,122.000000,122.000000,116.000000,122.000000
mean,7.213934,70.508197,68.508621,66.483607
std,12.299676,10.986277,12.735773,18.217798
min,0.000000,38.000000,37.000000,-1.000000
25%,3.625000,64.000000,60.000000,57.250000
50%,5.700000,70.000000,68.000000,68.000000
75%,7.900000,77.000000,77.000000,78.000000
max,99.000000,96.000000,100.000000,100.000000


In [20]:
students["program"].value_counts()

program
MDS    50
MIS    40
MCS    28
mds     4
Name: count, dtype: int64

Already we can see problems: `final` has a minimum of **-1**, `study_hours_per_week` a maximum of **99**, `midterm2` has missing values, and `program` contains both `MDS` and `mds`.

## 5. Cleaning

### Text and categories

In [21]:
students["program"] = students["program"].str.upper().str.strip()
students["program"].value_counts()

program
MDS    54
MIS    40
MCS    28
Name: count, dtype: int64

### Duplicates

In [22]:
print(students.duplicated().sum(), "duplicated rows")
students = students.drop_duplicates().reset_index(drop=True)
students["student_id"].is_unique

2 duplicated rows


True

### Invalid values

Sentinel values such as `-1` or `99` mean "unknown" and must become `NaN`, otherwise they distort every statistic.

In [23]:
print(students.loc[students["final"] < 0, ["student_id", "final"]])
students["final"] = students["final"].mask(students["final"] < 0)        # -> NaN
students.loc[students["study_hours_per_week"] > 80, "study_hours_per_week"] = np.nan
students.describe().loc[["min", "max"]]

   student_id  final
4    S2025004   -1.0
62   S2025062   -1.0
95   S2025095   -1.0


,study_hours_per_week,midterm1,midterm2,final
min,0.0,38.0,37.0,32.0
max,13.5,96.0,100.0,100.0


### Missing values

| Method | Effect |
|---|---|
| `isna()` / `notna()` | find missing values |
| `dropna()` | drop rows (or columns with `axis=1`) with missing values |
| `fillna(value)` | replace missing values |
| `interpolate()` | fill from neighbouring values (time series) |

In [24]:
students.isna().sum()

student_id              0
name                    0
gender                  0
program                 0
study_hours_per_week    2
midterm1                0
midterm2                6
final                   3
dtype: int64

In [25]:
# Option 1: drop students with any missing exam
print(len(students.dropna(subset=["midterm1", "midterm2", "final"])))

# Option 2: fill midterm2 with the median of the student's program
students["midterm2"] = students["midterm2"].fillna(
    students.groupby("program")["midterm2"].transform("median"))
students.isna().sum()

111


student_id              0
name                    0
gender                  0
program                 0
study_hours_per_week    2
midterm1                0
midterm2                0
final                   3
dtype: int64

> There is no universally right choice. Write down what you did and why; it changes the results.

### Types

In [26]:
students["program"] = students["program"].astype("category")
students["gender"] = students["gender"].astype("category")
students.dtypes

student_id                   str
name                         str
gender                  category
program                 category
study_hours_per_week     float64
midterm1                 float64
midterm2                 float64
final                    float64
dtype: object

## 6. Transforming

### New columns from existing ones

In [27]:
students["average"] = (0.3 * students["midterm1"] + 0.3 * students["midterm2"]
                       + 0.4 * students["final"]).round(1)
students["improved"] = students["final"] > students["midterm1"]
students.head()

,student_id,name,gender,program,study_hours_per_week,midterm1,midterm2,final,average,improved
0,S2025000,Sara,F,MDS,6.5,71.0,54.0,64.0,63.1,False
1,S2025001,Maha,F,MIS,4.5,73.0,85.0,71.0,75.8,False
2,S2025002,Sara,F,MIS,1.7,57.0,51.0,42.0,49.2,False
3,S2025003,Maha,F,MIS,3.6,71.0,67.0,48.0,60.6,False
4,S2025004,Lama,F,MIS,7.2,93.0,78.0,NaN,NaN,False


### Binning with `pd.cut`

In [28]:
students["letter"] = pd.cut(students["average"],
                            bins=[0, 60, 70, 80, 90, 100],
                            labels=["F", "D", "C", "B", "A"],
                            right=False)
students["letter"].value_counts().sort_index()

letter
F    21
D    43
C    33
B    14
A     6
Name: count, dtype: int64

### `map` and `apply`

- `Series.map(dict_or_function)`: transform each value of a column
- `DataFrame.apply(function, axis=0)`: apply a function to each column (`axis=1`: each row)
- `DataFrame.map(function)`: apply a function to every cell (called `applymap` before pandas 2.1)

In [29]:
students["gender_label"] = students["gender"].map({"F": "Female", "M": "Male"})

exams = students[["midterm1", "midterm2", "final"]]
print(exams.apply(lambda col: col.max() - col.min()))          # range per exam
print(exams.apply(lambda row: row.max() - row.min(), axis=1).head())

midterm1    58.0
midterm2    63.0
final       68.0
dtype: float64
0    17.0
1    14.0
2    15.0
3    23.0
4    15.0
dtype: float64


### Normalisation

Different columns are only comparable on a common scale.

| Method | Formula | Range |
|---|---|---|
| min-max | (x − min) / (max − min) | 0 to 1 |
| z-score | (x − mean) / std | mean 0, std 1 |
| mean scaling | x / mean | 1 = average |

In [30]:
def min_max(col):
    return (col - col.min()) / (col.max() - col.min())

def zscore(col):
    return (col - col.mean()) / col.std()

normalised = pd.concat({"min_max": exams.apply(min_max), "z": exams.apply(zscore)}, axis=1)
normalised.describe().round(2).loc[["mean", "std", "min", "max"]]

min_max                       z               
     midterm1 midterm2 final midterm1 midterm2 final
mean     0.56      0.5  0.53     0.00    -0.00 -0.00
std      0.19      0.2  0.22     1.00     1.00  1.00
min      0.00      0.0  0.00    -2.94    -2.52 -2.42
max      1.00      1.0  1.00     2.30     2.51  2.12

**Outliers**: a common rule flags values beyond 1.5 × IQR from the quartiles (the boxplot rule), or with |z| > 3.

In [31]:
q1, q3 = students["study_hours_per_week"].quantile([0.25, 0.75])
iqr = q3 - q1
outlier = (students["study_hours_per_week"] < q1 - 1.5 * iqr) | (students["study_hours_per_week"] > q3 + 1.5 * iqr)
students.loc[outlier, ["student_id", "study_hours_per_week"]]

,student_id,study_hours_per_week


## 7. Sorting and ranking

In [32]:
students.sort_values(["program", "average"], ascending=[True, False]).head()

,student_id,name,gender,program,study_hours_per_week,midterm1,midterm2,final,average,improved,letter,gender_label
19,S2025019,Omar,M,MCS,9.4,85.0,87.0,80.0,83.6,False,B,Male
35,S2025035,Majed,F,MCS,9.7,80.0,80.0,88.0,83.2,True,B,Female
25,S2025025,Abeer,M,MCS,4.0,78.0,68.0,77.0,74.6,False,C,Male
109,S2025109,Abdullah,M,MCS,6.0,65.0,65.0,86.0,73.4,True,C,Male
23,S2025023,Faisal,M,MCS,5.0,71.0,68.0,79.0,73.3,True,C,Male


In [33]:
students.nlargest(3, "average")[["name", "program", "average"]]

,name,program,average
105,Khalid,MDS,93.7
68,Reem,MDS,92.8
92,Mohammed,MDS,92.6


In [34]:
students["rank_in_program"] = students.groupby("program", observed=True)["average"].rank(ascending=False)
students[["name", "program", "average", "rank_in_program"]].head()

,name,program,average,rank_in_program
0,Sara,MDS,63.1,38.0
1,Maha,MIS,75.8,10.0
2,Sara,MIS,49.2,37.0
3,Maha,MIS,60.6,29.0
4,Lama,MIS,NaN,NaN


## 8. Grouping and aggregating

**Split-apply-combine**: split the rows into groups, apply a function to each group, combine the results.

In [35]:
students.groupby("program", observed=True)["average"].mean().round(1)

program
MCS    64.6
MDS    71.7
MIS    68.6
Name: average, dtype: float64

In [36]:
students.groupby(["program", "gender"], observed=True).agg(
    n=("student_id", "count"),
    mean_avg=("average", "mean"),
    best=("average", "max"),
    hours=("study_hours_per_week", "median"),
).round(1)

n  mean_avg  best  hours
program gender                           
MCS     F       16      62.6  83.2    5.8
        M       12      67.4  83.6    5.5
MDS     F       28      72.7  92.8    6.5
        M       25      70.6  93.7    5.8
MIS     F       21      67.7  87.7    4.4
        M       18      69.6  90.1    5.2

`transform` returns a result **aligned with the original rows**, which is handy for "compared with my group" features:

In [37]:
students["vs_program_mean"] = (students["average"]
                               - students.groupby("program", observed=True)["average"].transform("mean")).round(1)
students[["name", "program", "average", "vs_program_mean"]].head()

,name,program,average,vs_program_mean
0,Sara,MDS,63.1,-8.6
1,Maha,MIS,75.8,7.2
2,Sara,MIS,49.2,-19.4
3,Maha,MIS,60.6,-8.0
4,Lama,MIS,NaN,NaN


In [38]:
pd.crosstab(students["program"], students["letter"])

letter,F,D,C,B,A
program,,,,,
MCS,8,10,7,2,0
MDS,4,19,17,7,5
MIS,9,14,9,5,1


## 9. Combining tables

### `merge` (SQL-style joins)

In [39]:
courses

,program,program_name,credit_hours
0,MDS,Master of Data Science,36
1,MIS,Master of Information Systems,36
2,MCS,Master of Computer Science,33
3,MBA,Master of Business Administration,42


In [40]:
merged = students.merge(courses, on="program", how="left", validate="many_to_one")
merged[["name", "program", "program_name", "credit_hours"]].head()

,name,program,program_name,credit_hours
0,Sara,MDS,Master of Data Science,36
1,Maha,MIS,Master of Information Systems,36
2,Sara,MIS,Master of Information Systems,36
3,Maha,MIS,Master of Information Systems,36
4,Lama,MIS,Master of Information Systems,36


| `how=` | keeps |
|---|---|
| `"inner"` | only keys present in both tables |
| `"left"` | every row of the left table |
| `"right"` | every row of the right table |
| `"outer"` | every key from both |

`indicator=True` shows where each row came from, which is a good way to check a join:

In [41]:
check = students[["program"]].drop_duplicates().merge(courses, on="program", how="outer", indicator=True)
check[["program", "_merge"]]

,program,_merge
0,MBA,right_only
1,MCS,both
2,MDS,both
3,MIS,both


### `concat` (stacking)

In [42]:
semester1 = df.reset_index().assign(semester=1)
semester2 = df.reset_index().assign(semester=2)
pd.concat([semester1, semester2], ignore_index=True).shape

(10, 8)

## 10. Reshaping

**Wide** data has one column per measurement; **long** (tidy) data has one row per measurement. Plotting and grouping are usually easier in long form.

In [43]:
wide = students[["student_id", "midterm1", "midterm2", "final"]].head(3)
long = wide.melt(id_vars="student_id", var_name="exam", value_name="score")
long

,student_id,exam,score
0,S2025000,midterm1,71.0
1,S2025001,midterm1,73.0
2,S2025002,midterm1,57.0
3,S2025000,midterm2,54.0
4,S2025001,midterm2,85.0
5,S2025002,midterm2,51.0
6,S2025000,final,64.0
7,S2025001,final,71.0
8,S2025002,final,42.0


In [44]:
long.pivot(index="student_id", columns="exam", values="score")

exam,final,midterm1,midterm2
student_id,,,
S2025000,64.0,71.0,54.0
S2025001,71.0,73.0,85.0
S2025002,42.0,57.0,51.0


In [45]:
students.pivot_table(index="program", columns="gender", values="average",
                     aggfunc="mean", observed=True).round(1)

gender,F,M
program,,
MCS,62.6,67.4
MDS,72.7,70.6
MIS,67.7,69.6


### MultiIndex, `stack` and `unstack`

Grouping by two keys creates a hierarchical **MultiIndex**. `unstack` moves an index level to the columns; `stack` does the reverse.

In [46]:
by_two = students.groupby(["program", "gender"], observed=True)["average"].mean().round(1)
print(by_two)
by_two.unstack()

program  gender
MCS      F         62.6
         M         67.4
MDS      F         72.7
         M         70.6
MIS      F         67.7
         M         69.6
Name: average, dtype: float64


gender,F,M
program,,
MCS,62.6,67.4
MDS,72.7,70.6
MIS,67.7,69.6


## 11. Working with dates and times

In [47]:
dates = pd.Series(["01/03/2022", "15/03/2022", "02/04/2022"])
parsed = pd.to_datetime(dates, format="%d/%m/%Y")   # always state day-first formats explicitly!
parsed

0   2022-03-01
1   2022-03-15
2   2022-04-02
dtype: datetime64[us]

In [48]:
pd.DataFrame({
    "date": parsed,
    "year": parsed.dt.year,
    "month": parsed.dt.month_name(),
    "weekday": parsed.dt.day_name(),
    "is_weekend": parsed.dt.day_of_week.isin([4, 5]),   # Friday and Saturday in Saudi Arabia
})

,date,year,month,weekday,is_weekend
0,2022-03-01,2022,March,Tuesday,False
1,2022-03-15,2022,March,Tuesday,False
2,2022-04-02,2022,April,Saturday,True


A **DatetimeIndex** unlocks time-series tools: slicing by date strings, `resample` and `rolling`.

In [49]:
rng = np.random.default_rng(1)
daily = pd.Series(rng.poisson(15, 90),
                  index=pd.date_range("2022-01-01", periods=90, freq="D"), name="visits")
print(daily["2022-02"].head(3))
print(daily.resample("MS").sum())          # monthly totals
daily.rolling(7).mean().tail(3)            # 7-day moving average

2022-02-01    13
2022-02-02    17
2022-02-03    14
Freq: D, Name: visits, dtype: int64
2022-01-01    479
2022-02-01    406
2022-03-01    457
Freq: MS, Name: visits, dtype: int64


2022-03-29    11.571429
2022-03-30    12.714286
2022-03-31    11.571429
Freq: D, Name: visits, dtype: float64

## Summary

- `loc` selects by label, `iloc` by position, masks by condition.
- Inspect with `info`, `describe`, `value_counts`, `isna().sum()` before analysing.
- Turn sentinel values into `NaN`, then decide explicitly how to handle missing data.
- `groupby` + `agg`/`transform` and `merge` cover most everyday analysis.
- Convert dates with an explicit `format`, then use `.dt`, `resample` and `rolling`.

These are exactly the tools used in the Chapter 6 case study.